In [11]:
from pathlib import Path
from sklearn.datasets import make_blobs

import subprocess
import pandas as pd
import numpy as np
import os

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()

    for folder in [current, *current.parents]:
        if (folder / ".git").exists():
            return folder

    raise FileNotFoundError("No se encontró la raíz del proyecto")

PROJECT_ROOT = find_project_root()

INPUT_DIR = PROJECT_ROOT / "data" / "input"

OUTPUT_DIR_SERIAL = PROJECT_ROOT / "data" / "output"/"serial"
OUTPUT_DIR_PARALLEL = PROJECT_ROOT / "data" / "output"/"parallel_full"
OUTPUT_DIR_PARALLEL_K = PROJECT_ROOT / "data" / "output"/"parallel_k"

SRC_DIR = PROJECT_ROOT / "src"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_SERIAL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_PARALLEL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_PARALLEL_K.mkdir(parents=True, exist_ok=True)

In [12]:
# Indicate the number of points
n_points = [20000, 40000, 80000, 120000, 140000, 160000, 180000, 200000]

for n in n_points:
    # randomly generating data points and noise
    points, y_true = make_blobs(n_samples=n,
                                centers=4,
                                cluster_std=0.06,
                                random_state=11,
                                center_box=(0, 1.0))

    # only positive points and with three decimals
    points = np.round(np.abs(points[:, ::-1]), 3)

    input_file = INPUT_DIR / f"{n}_data.csv"

    # storing points into a csv file
    np.savetxt(input_file, points, delimiter=",",  fmt="%.3f")

In [13]:
epsilon = 0.03
min_puntos = 10

cores_virtuales = os.cpu_count()

num_hilos = [1, cores_virtuales//2, cores_virtuales, cores_virtuales*2]

def ejecutar_programa(programa, n, epsilon, min_puntos, num_hilos=None):
    comando = [str(SRC_DIR/programa), str(n), str(epsilon), str(min_puntos)]

    # El serial no recibe número de hilos.
    # parallel_full sí.
    if num_hilos is not None:
        comando.append(str(num_hilos))

    resultado = subprocess.run(comando, cwd=PROJECT_ROOT, capture_output=True, text=True)

    # Si el programa C++ terminó con error
    if resultado.returncode != 0:
        print(resultado.stdout)
        print(resultado.stderr)
        raise RuntimeError(
            f"Error al ejecutar {programa} con n={n}"
        )

    # Buscar la línea TIEMPO=...
    for linea in resultado.stdout.splitlines():
        if linea.startswith("TIEMPO="):
            return float(linea.split("=")[1])

In [21]:
resultados = []

num_puntos_prueba = [20000, 40000, 80000, 120000]
num_pruebas = 10

for n in num_puntos_prueba:

    print(f"Ejecutando n = {n}")

    tiempo_serial = []

    for prueba in range(num_pruebas):

        tiempo = ejecutar_programa("serial",n,epsilon,min_puntos)
        tiempo_serial.append(tiempo)

    tiempo_serial_promedio = sum(tiempo_serial)/num_pruebas

    for hilos in num_hilos:

        tiempo_parallel = []

        for prueba in range(num_pruebas):
    
            tiempo = ejecutar_programa("parallel_full",n,epsilon,min_puntos,hilos)
            tiempo_parallel.append(tiempo)
        
        tiempo_parallel_promedio = sum(tiempo_parallel)/num_pruebas
            
        speedup = tiempo_serial_promedio/tiempo_parallel_promedio

        resultados.append({
            "n_puntos": n,
            "hilos": hilos,
            "tiempo_serial": tiempo_serial_promedio,
            "tiempo_parallel_full": tiempo_parallel_promedio,
            "speedup": speedup
        })

df_resultados = pd.DataFrame(resultados)

df_resultados

Ejecutando n = 20000
Ejecutando n = 40000
Ejecutando n = 80000
Ejecutando n = 120000


,n_puntos,hilos,tiempo_serial,tiempo_parallel_full,speedup
0,20000,1,0.427087,0.377978,1.129925
1,20000,6,0.427087,0.160368,2.663176
2,20000,12,0.427087,0.131678,3.243413
3,20000,24,0.427087,0.131938,3.237022
4,40000,1,1.653964,1.822092,0.907728
5,40000,6,1.653964,0.758022,2.181948
6,40000,12,1.653964,0.592948,2.789393
7,40000,24,1.653964,0.555081,2.979683
8,80000,1,7.487144,7.255271,1.031959
9,80000,6,7.487144,3.067367,2.440903
